# ENERGY DATA CLEANING FROM WDI DATA SOURCE

Missing data were handled using a staged approach based on the proportion of missingness. First, variables with more than 50% missing values were removed from the dataset due to their high level of incompleteness. Second, numeric variables with low missingness (20% or less) were imputed using mean substitution. Third, numeric variables with moderate missingness (between 20% and 50%) were imputed using K-Nearest Neighbors (KNN) imputation with 5 neighbors. Complete numeric variables and those already mean-imputed were included as predictors in the KNN procedure to improve the quality of imputation.

In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
from sklearn.impute import KNNImputer

# Load your data
df = pd.read_excel("RAW_ENERGY_DATA.xlsx")

# Quick check
print("Initial shape:", df.shape)
df.head()


Initial shape: (1242, 72)


,Country Name,Time,Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Adjusted savings: energy depletion (% of GNI),Adjusted savings: natural resources depletion (% of GNI),...,Renewable energy consumption (% of total final energy consumption),Renewable internal freshwater resources per capita (cubic meters),"Renewable internal freshwater resources, total (billion cubic meters)",Rural population,Rural population (% of total population),Rural population growth (annual %),Time to obtain an electrical connection (days),Urban population,Urban population (% of total population),Urban population growth (annual %)
0,Angola,2022,50.0,8.6,75.9,48.5,NaN,76.2,NaN,NaN,...,NaN,4153.216769,148.00,11374345,31.919,1.216165,NaN,24260684,68.081,4.059358
1,Benin,2022,6.0,5.1,6.5,56.5,45.5,71.1,NaN,NaN,...,NaN,748.573658,10.30,6943870,50.466,1.439953,NaN,6815631,49.534,3.688459
2,Botswana,2022,66.0,25.3,86.9,75.9,25.0,95.5,NaN,NaN,...,NaN,983.650096,2.40,677704,27.776,-0.774032,NaN,1762188,72.224,2.512128
3,Burkina Faso,2022,17.2,3.0,47.8,19.5,3.4,60.5,NaN,NaN,...,71.4,555.332485,12.50,15333832,68.123,1.378344,NaN,7175206,31.877,4.327612
4,Burundi,2022,0.1,0.1,0.2,10.3,1.6,64.0,NaN,NaN,...,83.0,755.193060,10.06,11400594,85.583,2.287245,NaN,1920503,14.417,5.227534


In [3]:
df.describe()

,Time,Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Adjusted savings: energy depletion (% of GNI),Adjusted savings: natural resources depletion (% of GNI),Adjusted savings: particulate emission damage (% of GNI),...,Renewable energy consumption (% of total final energy consumption),Renewable internal freshwater resources per capita (cubic meters),"Renewable internal freshwater resources, total (billion cubic meters)",Rural population,Rural population (% of total population),Rural population growth (annual %),Time to obtain an electrical connection (days),Urban population,Urban population (% of total population),Urban population growth (annual %)
count,1242.000000,1219.000000,1219.000000,1219.000000,1226.000000,1085.000000,1235.000000,1141.000000,1130.000000,1120.000000,...,1200.000000,1195.000000,1195.000000,1.242000e+03,1242.000000,1241.000000,99.000000,1.242000e+03,1242.000000,1241.000000
mean,2011.000000,25.933962,16.671411,36.525882,44.812153,31.011705,68.371984,3.492557,8.216224,1.871021,...,59.969000,8127.772713,75.342819,1.236669e+07,57.344566,1.399129,38.620884,8.251551e+06,42.655434,3.523352
std,6.635922,33.056736,30.747687,37.040860,29.474239,31.707117,24.427494,9.075130,10.313102,1.238352,...,30.300219,16915.693013,141.794842,1.834424e+07,18.022099,1.292978,35.663402,1.357184e+07,18.022099,1.551738
min,2000.000000,0.000000,0.000000,0.000000,0.800000,0.500000,3.500000,0.000000,0.000000,0.144684,...,0.100000,8.879555,0.300000,4.004500e+04,9.265000,-5.730342,4.500000,4.091700e+04,8.246000,-4.431066
25%,2005.000000,1.200000,0.400000,3.000000,18.500000,5.700000,51.150000,0.000000,1.287160,1.024864,...,34.150000,826.968366,3.750000,1.351316e+06,44.710000,0.599992,16.105038,1.213013e+06,29.080750,2.567055
50%,2011.000000,8.100000,1.350000,19.300000,41.550000,19.700000,72.400000,0.006193,4.710336,1.655644,...,71.850000,1525.595682,16.000000,7.434410e+06,58.775000,1.626154,25.458801,3.435788e+06,41.225000,3.653954
75%,2017.000000,40.500000,13.550000,76.050000,65.000000,42.100000,88.050000,1.550356,10.756329,2.459063,...,84.600000,5810.486779,92.150000,1.334948e+07,70.919250,2.178716,48.754626,9.269055e+06,55.290000,4.500676
max,2022.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,94.073122,102.890683,8.975804,...,98.300000,128569.771052,900.000000,1.037183e+08,91.754000,7.310198,194.342453,1.194326e+08,90.735000,9.893099


In [8]:
# Percentage of missing values per column
missing_percent = df.isna().mean() * 100

missing_summary = (
    missing_percent
    .reset_index()
    .rename(columns={'index': 'Variable', 0: 'Percent_Missing'})
    .sort_values(by='Percent_Missing', ascending=False)
)

print("Missingness summary (sorted):")
missing_summary


Missingness summary (sorted):


,Variable,Percent_Missing
32,"Energy imports, net (% of energy use)",44.927536
46,"Inflation, consumer prices (annual %)",35.909823
23,Electric power transmission and distribution l...,34.219002
43,GDP per unit of energy use (constant 2021 PPP ...,32.930757
35,"Energy use (kg of oil equivalent) per $1,000 G...",32.930757
...,...,...
53,"Population, total",0.000000
60,Rural population (% of total population),0.000000
59,Rural population,0.000000
62,Urban population,0.000000


In [9]:
# Define groups based on % missing
high_missing_cols = missing_percent[missing_percent > 50].index.tolist()
moderate_missing_cols = missing_percent[(missing_percent > 20) & (missing_percent <= 50)].index.tolist()
low_missing_cols = missing_percent[(missing_percent > 5) & (missing_percent <= 20)].index.tolist()
very_low_missing_cols = missing_percent[(missing_percent > 0) & (missing_percent <= 5)].index.tolist()
no_missing_cols = missing_percent[missing_percent == 0].index.tolist()

print("High missing (>50%):", len(high_missing_cols))
print("Moderate missing (20–50%):", len(moderate_missing_cols))
print("Low missing (5–20%):", len(low_missing_cols))
print("Very low missing (0–5%):", len(very_low_missing_cols))
print("No missing (0%):", len(no_missing_cols))


High missing (>50%): 0
Moderate missing (20–50%): 12
Low missing (5–20%): 19
Very low missing (0–5%): 27
No missing (0%): 7


In [10]:
# Drop variables with more than 50% missing values
df = df.drop(columns=high_missing_cols)

print("Shape after dropping high-missing columns:", df.shape)

# Identify numeric columns after dropping
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols


Shape after dropping high-missing columns: (1242, 65)


['Time',
 'Access to clean fuels and technologies for cooking (% of population)',
 'Access to clean fuels and technologies for cooking, rural (% of rural population)',
 'Access to clean fuels and technologies for cooking, urban (% of urban population)',
 'Access to electricity (% of population)',
 'Access to electricity, rural (% of rural population)',
 'Access to electricity, urban (% of urban population)',
 'Adjusted savings: energy depletion (% of GNI)',
 'Adjusted savings: natural resources depletion (% of GNI)',
 'Adjusted savings: particulate emission damage (% of GNI)',
 'Adjusted savings: particulate emission damage (current US$)',
 'Alternative and nuclear energy (% of total energy use)',
 'Carbon dioxide (CO2) emissions (total) excluding LULUCF (% change from 1990)',
 'Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e)',
 'Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)',
 'Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)',
 'C

In [11]:
# Keep only numeric variables in each group
low_numeric = [c for c in low_missing_cols if c in numeric_cols]
moderate_numeric = [c for c in moderate_missing_cols if c in numeric_cols]
very_low_numeric = [c for c in very_low_missing_cols if c in numeric_cols]

print("Low-missing numeric (5–20%):", low_numeric)
print("Moderate-missing numeric (20–50%):", moderate_numeric)
print("Very-low-missing numeric (0–5%):", very_low_numeric)


Low-missing numeric (5–20%): ['Access to electricity, rural (% of rural population)', 'Adjusted savings: energy depletion (% of GNI)', 'Adjusted savings: natural resources depletion (% of GNI)', 'Adjusted savings: particulate emission damage (% of GNI)', 'Adjusted savings: particulate emission damage (current US$)', 'Consumer price index (2010 = 100)', 'Debt service (PPG and IMF only, % of exports of goods, services and primary income)', 'Debt service on external debt, public and publicly guaranteed (PPG) (TDS, current US$)', 'Debt service on external debt, total (TDS, current US$)', 'Electricity production from nuclear sources (% of total)', 'Electricity production from renewable sources, excluding hydroelectric (% of total)', 'External debt stocks (% of GNI)', 'Final consumption expenditure (% of GDP)', 'Government Effectiveness: Estimate', 'Natural gas rents (% of GDP)', 'Net primary income (BoP, current US$)', 'Oil rents (% of GDP)', 'Regulatory Quality: Estimate', 'Renewable elect

In [12]:
# Combine very-low and low missing into one group for mean imputation
mean_impute_numeric = sorted(list(set(low_numeric + very_low_numeric)))

print("Numeric variables with <=20% missing to mean-impute:", mean_impute_numeric)

for col in mean_impute_numeric:
    df[col] = df[col].fillna(df[col].mean())

print("Done: mean imputation for numeric variables with <=20% missing.")


Numeric variables with <=20% missing to mean-impute: ['Access to clean fuels and technologies for cooking (% of population)', 'Access to clean fuels and technologies for cooking, rural (% of rural population)', 'Access to clean fuels and technologies for cooking, urban (% of urban population)', 'Access to electricity (% of population)', 'Access to electricity, rural (% of rural population)', 'Access to electricity, urban (% of urban population)', 'Adjusted savings: energy depletion (% of GNI)', 'Adjusted savings: natural resources depletion (% of GNI)', 'Adjusted savings: particulate emission damage (% of GNI)', 'Adjusted savings: particulate emission damage (current US$)', 'Carbon dioxide (CO2) emissions (total) excluding LULUCF (% change from 1990)', 'Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e)', 'Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)', 'Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)', 'Carbon intensity of GDP (kg CO

In [14]:
# Columns used in KNN imputation:
#  variables with moderate missingness (to be imputed)
#  plus already-complete or mean-imputed numeric columns to help the model
knn_cols = list(set(moderate_numeric + mean_impute_numeric + no_missing_cols))
knn_cols = [c for c in knn_cols if c in numeric_cols]  # ensure all exist and are numeric

print("Columns used in KNN imputation:", knn_cols)

if len(moderate_numeric) > 0 and len(knn_cols) > 0:
    imputer = KNNImputer(n_neighbors=5)
    
    knn_data = df[knn_cols]
    imputed_array = imputer.fit_transform(knn_data)
    
    imputed_knn = pd.DataFrame(imputed_array, columns=knn_cols, index=df.index)
    
    # Overwrite only the moderate-missing columns with KNN-imputed values
    df[moderate_numeric] = imputed_knn[moderate_numeric]
    
    print("Done: KNN imputation for numeric variables with 20–50% missing.")
else:
    print("No moderate-missing numeric variables to impute with KNN.")


Columns used in KNN imputation: ['External debt stocks (% of GNI)', 'Electricity production from oil, gas and coal sources (% of total)', 'Net primary income (BoP, current US$)', 'Adjusted savings: natural resources depletion (% of GNI)', 'Access to clean fuels and technologies for cooking, rural (% of rural population)', 'Electricity production from renewable sources, excluding hydroelectric (kWh)', 'Fuel imports (% of merchandise imports)', 'Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)', 'Population growth (annual %)', 'Consumer price index (2010 = 100)', 'Population, total', 'Renewable energy consumption (% of total final energy consumption)', 'Oil rents (% of GDP)', 'Renewable electricity output (% of total electricity output)', 'Debt service on external debt, public and publicly guaranteed (PPG) (TDS, current US$)', 'Electricity production from hydroelectric sources (% of total)', 'Adjusted savings: energy depletion (% of GNI)', 'Urban population', 'Alternative a

In [15]:
remaining_missing = df.isna().mean() * 100

print("Remaining variables with any missingness (>0%):")
print(remaining_missing[remaining_missing > 0])

print("\nFinal shape of df:", df.shape)


Remaining variables with any missingness (>0%):
Series([], dtype: float64)

Final shape of df: (1242, 65)


In [16]:
# For logging: note that mean_impute_numeric covers 0–20% missing
log_data = {
    "Dropped_Variables_(>50%_missing)": high_missing_cols,
    "Mean_Imputed_(0-20%_missing_numeric)": mean_impute_numeric,
    "KNN_Imputed_(20-50%_missing_numeric)": moderate_numeric,
    "No_Missing_(0%)_Not_Imputed": [c for c in no_missing_cols if c in df.columns]
}

log_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in log_data.items()]))

print("\n==== VARIABLE TREATMENT LOG ====")
print(log_df)

log_df.to_csv("missingness_treatment_log_revised.csv", index=False)
print("\nLog saved as 'missingness_treatment_log_revised.csv'")



==== VARIABLE TREATMENT LOG ====
   Dropped_Variables_(>50%_missing)  \
0                               NaN   
1                               NaN   
2                               NaN   
3                               NaN   
4                               NaN   
5                               NaN   
6                               NaN   
7                               NaN   
8                               NaN   
9                               NaN   
10                              NaN   
11                              NaN   
12                              NaN   
13                              NaN   
14                              NaN   
15                              NaN   
16                              NaN   
17                              NaN   
18                              NaN   
19                              NaN   
20                              NaN   
21                              NaN   
22                              NaN   
23                            

In [17]:
# Save the cleaned dataset as an Excel file
df.to_excel("ENERGY_POVERTY_CLEANED.xlsx", index=False)

print("Cleaned dataset saved as 'ENERGY_POVERTY_CLEANED.xlsx'")


Cleaned dataset saved as 'ENERGY_POVERTY_CLEANED.xlsx'
